# 01 — ShuttleSet EDA + Train/Val Split

Loads directly from `ShuttleSet/set/{match}/*.csv` — no output JSONs needed.  
**4 matches excluded** (wrong broadcast angle: An Se Young/Pornpawee, CHEN Long/CHOU Denmark Open, CHOU/Jonatan Sudirman, CHOU/NG Sudirman) → moved to `others_dataset_excluded/`.

1. Coverage — which of the 44 matches have frames extracted? (38 usable)
2. Shot type distribution across all 38 usable matches
3. Per-match statistics (strokes, rallies, sets)
4. Rally length distribution
5. Shot transition matrix
6. Court position heatmaps
7. Annotation completeness
8. Tournament & player breakdown
9. **Train/Val split** — match-level split with shot-type balance analysis → symlinks in `shuttleset_frames/train/` and `shuttleset_frames/val/`

In [1]:
import sys
sys.path.insert(0, '..')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from src.config import SS_CSV_ROOT, SS_MATCH_CSV, SS_FRAMES

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

SHOT_TYPE_EN = {
    '發短球': 'Short Serve',    '發長球': 'Long Serve',
    '推球':   'Push',           '撲球':   'Rush',
    '殺球':   'Smash',          '點扣':   'Tap Smash',
    '切球':   'Slice',          '過度切球': 'Trans. Slice',
    '長球':   'Clear',          '平球':   'Drive',
    '後場抽平球': 'BG Drive',   '小平球': 'Half Smash',
    '擋小球': 'Net Block',      '放小球': 'Net Drop',
    '勾球':   'Cross-Net',      '挑球':   'Lob',
    '防守回抽': 'Def. Drive',   '防守回挑': 'Def. Lob',
    '未知球種': 'Unknown',
}

print("Config loaded.")

ModuleNotFoundError: No module named 'src'

## 1. Coverage — which matches have frames extracted?

In [ ]:
EXCLUDED = {
    'An_Se_Young_Pornpawee_Chochuwong_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
    'CHEN_Long_CHOU_Tien_Chen_Denmark_Open_2019_QuarterFinal',
    'CHOU_Tien_Chen_Jonatan_CHRISTIE_Sudirman_Cup_2019_Quarter-finals',
    'CHOU_Tien_Chen_NG_Ka_Long_Angus_Sudirman_Cup_2019_Group_Stage',
}

match_df = pd.read_csv(SS_MATCH_CSV)
match_df['video'] = match_df['video'].str.strip()

# Only dirs that contain actual frame jpg files (excludes train/val aggregation dirs)
frames_root = Path(SS_FRAMES)
frame_counts = {}
for d in frames_root.iterdir():
    if d.is_dir():
        n = sum(1 for _ in d.glob('frame_*.jpg'))
        if n > 0:
            frame_counts[d.name] = n

# Remove excluded matches from frame_counts
for m in EXCLUDED:
    frame_counts.pop(m, None)

all_match_ids = set(match_df['video']) - EXCLUDED
has_frames     = set(frame_counts.keys())
no_frames      = all_match_ids - has_frames

match_df = match_df[~match_df['video'].isin(EXCLUDED)].copy()
match_df['has_frames']  = match_df['video'].isin(has_frames)
match_df['frame_count'] = match_df['video'].map(frame_counts).fillna(0).astype(int)

print(f"Total matches in match.csv (excl. 4 wrong-angle): {len(match_df)}")
print(f"Matches with frames                             : {len(has_frames)}")
print(f"No folder at all                                : {len(no_frames)} → {sorted(no_frames)}")
print(f"\nUsable (has frames + CSV)  : {len(has_frames)}")

fig, ax = plt.subplots(figsize=(5, 5))
sizes  = [len(has_frames), len(no_frames)]
labels = [f'Has frames\n({sizes[0]})', f'No folder\n({sizes[1]})']
colors = ['#2ecc71', '#e74c3c']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
ax.set_title(f'Frame Extraction Coverage ({len(match_df)} matches, 4 excluded)')
plt.tight_layout()
plt.show()

## 2. Load All CSV Annotations (38 usable matches)

In [ ]:
csv_root = Path(SS_CSV_ROOT)
all_records = []

for match_id in sorted(has_frames):
    match_csv_dir = csv_root / match_id
    if not match_csv_dir.exists():
        print(f"  [WARN] No CSV dir: {match_id}")
        continue
    for csv_path in sorted(match_csv_dir.glob('set*.csv')):
        df = pd.read_csv(csv_path)
        df['match_id'] = match_id
        df['set_file'] = csv_path.name
        all_records.append(df)

strokes_df = pd.concat(all_records, ignore_index=True)
strokes_df.columns = strokes_df.columns.str.strip()

strokes_df['frame_path'] = strokes_df.apply(
    lambda r: str(frames_root / r['match_id'] / f"frame_{int(r['frame_num']):06d}.jpg"),
    axis=1
)

print(f"Total stroke records : {len(strokes_df)}")
print(f"Unique matches       : {strokes_df['match_id'].nunique()}")
print(f"Unique shot types    : {strokes_df['type'].nunique()}")
print(f"Columns              : {list(strokes_df.columns)}")
strokes_df.head(3)

## 3. Shot Type Distribution

In [ ]:
type_counts = strokes_df['type'].value_counts()

fig, ax = plt.subplots(figsize=(14, 5))
en_labels = [f"{SHOT_TYPE_EN.get(s, s)}\n({s})" for s in type_counts.index]
colors = plt.cm.tab20(np.linspace(0, 1, len(type_counts)))
bars = ax.bar(range(len(type_counts)), type_counts.values, color=colors)
ax.set_xticks(range(len(type_counts)))
ax.set_xticklabels(en_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Count')
ax.set_title(f'Shot Type Distribution  (n={len(strokes_df):,} strokes, {strokes_df["match_id"].nunique()} matches)')
for bar, cnt in zip(bars, type_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(cnt), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

print(f"\nImbalance ratio (max/min): {type_counts.max() / type_counts.min():.1f}×")
for shot, cnt in type_counts.items():
    print(f"  {SHOT_TYPE_EN.get(shot, shot):>15} ({shot}): {cnt:>5}  {cnt/len(strokes_df)*100:>5.1f}%")

## 4. Per-Match Statistics

In [ ]:
match_stats = strokes_df.groupby('match_id').agg(
    n_strokes=('type', 'count'),
    n_rallies=('rally', 'nunique'),
    n_sets=('set_file', 'nunique'),
).sort_values('n_strokes', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(range(len(match_stats)), match_stats['n_strokes'], color='#3498db')
axes[0].set_yticks(range(len(match_stats)))
axes[0].set_yticklabels([m.replace('_', ' ')[:38] for m in match_stats.index], fontsize=6)
axes[0].set_xlabel('Number of strokes')
axes[0].set_title('Strokes per Match')
axes[0].invert_yaxis()

axes[1].hist(match_stats['n_strokes'], bins=15, color='#2ecc71', edgecolor='black')
axes[1].axvline(match_stats['n_strokes'].mean(), color='red', linestyle='--',
                label=f"Mean = {match_stats['n_strokes'].mean():.0f}")
axes[1].set_xlabel('Strokes per match')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Strokes per Match')
axes[1].legend()

plt.tight_layout()
plt.show()

print(match_stats.describe().round(1))

## 5. Rally Length Distribution

In [ ]:
rally_len = strokes_df.groupby(['match_id', 'rally']).size().reset_index(name='n_shots')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rally_len['n_shots'], bins=range(1, rally_len['n_shots'].max() + 2),
             color='#9b59b6', edgecolor='black', alpha=0.8)
axes[0].axvline(rally_len['n_shots'].mean(), color='red', linestyle='--',
                label=f"Mean = {rally_len['n_shots'].mean():.1f}")
axes[0].axvline(rally_len['n_shots'].median(), color='orange', linestyle='--',
                label=f"Median = {rally_len['n_shots'].median():.0f}")
axes[0].set_xlabel('Shots per rally')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Rally Length Distribution')
axes[0].legend()

rallies_per_match = rally_len.groupby('match_id').size().sort_values(ascending=False)
axes[1].barh(range(len(rallies_per_match)), rallies_per_match.values, color='#e67e22')
axes[1].set_yticks(range(len(rallies_per_match)))
axes[1].set_yticklabels([m.replace('_', ' ')[:35] for m in rallies_per_match.index], fontsize=6)
axes[1].set_xlabel('Rallies')
axes[1].set_title('Rallies per Match')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Total rallies: {len(rally_len)}")
print(f"Shots/rally — min: {rally_len['n_shots'].min()}, "
      f"median: {rally_len['n_shots'].median():.0f}, "
      f"max: {rally_len['n_shots'].max()}")

## 6. Shot Transition Matrix

In [ ]:
transitions = Counter()
for (match_id, rally), grp in strokes_df.groupby(['match_id', 'rally']):
    shots = grp.sort_values('ball_round')['type'].tolist()
    for a, b in zip(shots, shots[1:]):
        transitions[(a, b)] += 1

all_types = type_counts.index.tolist()
trans_mat = pd.DataFrame(0, index=all_types, columns=all_types)
for (a, b), cnt in transitions.items():
    if a in trans_mat.index and b in trans_mat.columns:
        trans_mat.loc[a, b] = cnt
trans_prob = trans_mat.div(trans_mat.sum(axis=1), axis=0).fillna(0)

en_idx = [SHOT_TYPE_EN.get(s, s) for s in all_types]
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(trans_prob, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=en_idx, yticklabels=en_idx,
            ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_xlabel('Next Shot')
ax.set_ylabel('Current Shot')
ax.set_title('Shot Transition Probabilities (row-normalised)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Court Position Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

def heatmap2d(ax, x, y, title, cmap):
    mask = x.notna() & y.notna()
    if mask.sum() > 0:
        h = ax.hist2d(x[mask], y[mask], bins=30, cmap=cmap)
        plt.colorbar(h[3], ax=ax)
    ax.set_title(f'{title} (n={mask.sum():,})')
    ax.invert_yaxis(); ax.set_aspect('equal')

heatmap2d(axes[0], strokes_df['hit_x'],      strokes_df['hit_y'],      'Hit Positions',     'Blues')
heatmap2d(axes[1], strokes_df['landing_x'],  strokes_df['landing_y'],  'Landing Positions', 'Reds')

# Player vs opponent
pm = strokes_df['player_location_x'].notna()
om = strokes_df['opponent_location_x'].notna()
axes[2].scatter(strokes_df.loc[pm, 'player_location_x'], strokes_df.loc[pm, 'player_location_y'],
                alpha=0.15, s=4, c='green', label='Hitter')
axes[2].scatter(strokes_df.loc[om, 'opponent_location_x'], strokes_df.loc[om, 'opponent_location_y'],
                alpha=0.15, s=4, c='orange', label='Opponent')
axes[2].set_title('Player Positions at Hit')
axes[2].invert_yaxis(); axes[2].set_aspect('equal')
axes[2].legend(markerscale=5)

plt.tight_layout()
plt.show()

## 8. Annotation Completeness

In [ ]:
feat_cols = ['hit_area', 'hit_x', 'hit_y', 'landing_area', 'landing_x', 'landing_y',
             'player_location_x', 'player_location_y',
             'opponent_location_x', 'opponent_location_y',
             'backhand', 'hit_height']

miss_pct = (strokes_df[feat_cols].isna().mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if p > 50 else '#e67e22' if p > 20 else '#2ecc71' for p in miss_pct]
ax.barh(miss_pct.index, miss_pct.values, color=colors)
ax.set_xlabel('% Missing')
ax.set_title('Annotation Completeness')
for i, (col, pct) in enumerate(miss_pct.items()):
    ax.text(pct + 0.5, i, f'{pct:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Tournament & Player Breakdown

In [ ]:
meta = match_df[match_df['has_frames']][['video', 'tournament', 'round', 'year', 'winner', 'loser']].copy()
meta = meta.rename(columns={'video': 'match_id'})
strokes_meta = strokes_df.merge(meta, on='match_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tourn = strokes_meta['tournament'].value_counts()
axes[0].barh(range(len(tourn)), tourn.values, color='#3498db')
axes[0].set_yticks(range(len(tourn)))
axes[0].set_yticklabels(tourn.index, fontsize=8)
axes[0].set_xlabel('Strokes')
axes[0].set_title('Strokes by Tournament')
axes[0].invert_yaxis()

rnd = strokes_meta['round'].value_counts()
axes[1].bar(range(len(rnd)), rnd.values, color='#9b59b6')
axes[1].set_xticks(range(len(rnd)))
axes[1].set_xticklabels(rnd.index, rotation=30, ha='right')
axes[1].set_ylabel('Strokes')
axes[1].set_title('Strokes by Round')

plt.tight_layout()
plt.show()

all_players = pd.concat([meta['winner'], meta['loser']]).value_counts()
print(f"\nUnique players: {len(all_players)}")
print(all_players.to_string())

DECIDE TO REDUCE TO 20 MATCHES FOR TRAINING 



## 10. Shot Type per Match Heatmap

In [ ]:
spm = strokes_df.groupby(['match_id', 'type']).size().unstack(fill_value=0)
spm.columns = [SHOT_TYPE_EN.get(c, c) for c in spm.columns]
spm.index = [m.replace('_', ' ')[:35] for m in spm.index]

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(spm, annot=True, fmt='d', cmap='YlGnBu',
            ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Shot Type Counts per Match')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

## 11. Train / Val Split Design

**Strategy:** Split at **match level** (whole matches → train or val).  
- Granularity: match (not rally) — avoids frame leakage within a match and means we can symlink whole folders.  
- Target: **~80 % train / ~20 % val** → ~33 train, ~8 val.  
- Balancing criterion: minimise KL-divergence between train and val shot-type distributions.  
- Algorithm: greedy assignment — sort matches by size (largest first), greedily assign each to the split that would most improve overall balance.

In [ ]:
import numpy as np

VAL_RATIO = 0.20   # target fraction of strokes in val

# Per-match shot type distribution (normalised)
match_type_vec = (
    strokes_df.groupby(['match_id', 'type']).size()
    .unstack(fill_value=0)
    .reindex(columns=type_counts.index, fill_value=0)
)
match_type_vec_norm = match_type_vec.div(match_type_vec.sum(axis=1), axis=0)

# Greedy balanced split
match_sizes = strokes_df.groupby('match_id').size().sort_values(ascending=False)
total = match_sizes.sum()
target_val = total * VAL_RATIO

train_ids, val_ids = [], []
train_vec = np.zeros(len(type_counts))
val_vec   = np.zeros(len(type_counts))
val_total = 0

for match_id, size in match_sizes.items():
    if val_total + size <= target_val * 1.15:   # 15% slack
        val_ids.append(match_id)
        val_vec   += match_type_vec.loc[match_id].values
        val_total += size
    else:
        train_ids.append(match_id)
        train_vec += match_type_vec.loc[match_id].values

print(f"Train: {len(train_ids)} matches, {train_vec.sum():.0f} strokes "
      f"({train_vec.sum()/total*100:.1f}%)")
print(f"Val  : {len(val_ids)}  matches, {val_vec.sum():.0f} strokes "
      f"({val_vec.sum()/total*100:.1f}%)")

print(f"\nVal matches:")
for m in sorted(val_ids):
    print(f"  {m}")

In [ ]:
# Visual: compare shot type distribution between train and val
train_norm = train_vec / train_vec.sum()
val_norm   = val_vec   / val_vec.sum()
en_types   = [SHOT_TYPE_EN.get(t, t) for t in type_counts.index]

x = np.arange(len(type_counts))
w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, train_norm, w, label='Train', color='#3498db', alpha=0.85)
ax.bar(x + w/2, val_norm,   w, label='Val',   color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(en_types, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Proportion')
ax.set_title('Shot Type Distribution: Train vs Val')
ax.legend()
plt.tight_layout()
plt.show()

# KL divergence (lower = better balance)
eps = 1e-9
kl = (train_norm * np.log((train_norm + eps) / (val_norm + eps))).sum()
print(f"KL(train ‖ val) = {kl:.4f}  (lower is more balanced)")

## 12. Execute Split — Create Symlinks

Creates `shuttleset_frames/train/{match_id}` and `shuttleset_frames/val/{match_id}` as **symlinks** pointing to the original match folder. No data is copied.

In [ ]:
train_dir = frames_root / 'train'
val_dir   = frames_root / 'val'
train_dir.mkdir(exist_ok=True)
val_dir.mkdir(exist_ok=True)

def make_symlink(split_dir, match_id):
    link = split_dir / match_id
    target = frames_root / match_id   # absolute target
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(target)

for m in train_ids:
    make_symlink(train_dir, m)
for m in val_ids:
    make_symlink(val_dir, m)

print(f"Created {len(train_ids)} symlinks in {train_dir}")
print(f"Created {len(val_ids)}  symlinks in {val_dir}")
print("\nVal symlinks:")
for p in sorted(val_dir.iterdir()):
    print(f"  {p.name} → {p.resolve().name}")

## 13. Save Split Manifest + Summary

In [ ]:
import json

manifest = {
    'train': sorted(train_ids),
    'val':   sorted(val_ids),
    'stats': {
        'total_matches': len(train_ids) + len(val_ids),
        'train_matches': len(train_ids),
        'val_matches':   len(val_ids),
        'train_strokes': int(train_vec.sum()),
        'val_strokes':   int(val_vec.sum()),
        'kl_divergence': float(kl),
    }
}

manifest_path = frames_root.parent / 'shuttleset_split.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"Saved split manifest → {manifest_path}")

# Final summary
print("\n" + "="*60)
print("SHUTTLESET EDA SUMMARY")
print("="*60)
print(f"Matches total         : {len(match_df)}")
print(f"  With frames (usable): {len(has_frames)}")
print(f"  Empty folders       : {len(empty_frames)}")
print(f"  No folder           : {len(no_frames)}")
print(f"Total strokes         : {len(strokes_df):,}")
print(f"Total rallies         : {len(rally_len):,}")
print(f"Unique shot types     : {strokes_df['type'].nunique()}")
print(f"Avg shots/rally       : {rally_len['n_shots'].mean():.1f}")
print(f"Imbalance ratio       : {type_counts.max()/type_counts.min():.0f}×")
print(f"Train split           : {len(train_ids)} matches  {train_vec.sum()/total*100:.1f}% strokes")
print(f"Val   split           : {len(val_ids)}  matches  {val_vec.sum()/total*100:.1f}% strokes")
print(f"KL(train‖val)         : {kl:.4f}")
print("="*60)